# PR0502. Manipulación básica de dataframes

In [40]:
from pyspark.sql import SparkSession

try: 
    spark = (SparkSession.builder.appName("PR0502")
              .master("spark://spark-master:7077")
              .getOrCreate()
            )

    print("SparkSession iniciada correctamente.")
except Exception as e:
    print("Error en la conexion")
    print(e)

sc = spark.sparkContext

SparkSession iniciada correctamente.


## Dataset 1: Datos para la predicción del rendimiento en cultivos

### 1.- Selección de características

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType
from pyspark.sql.functions import col

schema_crop = StructType([
    StructField("Crop", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Soil_Type", StringType(), True),
    StructField("Soil_pH", DoubleType(), True),
    StructField("Rainfall_mm", DoubleType(), True),
    StructField("Temperature_C", DoubleType(), True),
    StructField("Humidity_pct", DoubleType(), True),
    StructField("Fertilizer_Used_kg", DoubleType(), True),
    StructField("Irrigation", StringType(), True),
    StructField("Pesticides_Used_kg", DoubleType(), True),
    StructField("Planting_Density", DoubleType(), True),
    StructField("Previous_Crop", StringType(), True),
    StructField("Yield_ton_per_ha", DoubleType(), True)
])

df_sel = (spark.read
                .format("csv")
                .schema(schema_crop)
                .option("header", "true")
                .option("quote", "\"")
                .load("./crop_yield_dataset.csv")
         )

df_sel = df_sel.select("Crop", "Region", "Rainfall_mm", "Temperature_C", "Irrigation", "Yield_ton_per_ha")

df_sel.printSchema()
df_sel.show(5)

root
 |-- Crop: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Rainfall_mm: double (nullable = true)
 |-- Temperature_C: double (nullable = true)
 |-- Irrigation: string (nullable = true)
 |-- Yield_ton_per_ha: double (nullable = true)

+------+--------+-----------+-------------+----------+----------------+
|  Crop|  Region|Rainfall_mm|Temperature_C|Irrigation|Yield_ton_per_ha|
+------+--------+-----------+-------------+----------+----------------+
| Maize|Region_C|     1485.4|         19.7|      Drip|          101.48|
|Barley|Region_D|      399.4|         29.1| Sprinkler|          127.39|
|  Rice|Region_C|      980.9|         30.5| Sprinkler|           68.99|
| Maize|Region_D|     1054.3|         26.4|      Drip|          169.06|
| Maize|Region_D|      744.6|         20.4|      Drip|          118.71|
+------+--------+-----------+-------------+----------+----------------+
only showing top 5 rows



### 2.- Normalización de nombres

In [10]:
df_renamed = df_sel.withColumnRenamed("Temperature_C", "Temperatura") \
                   .withColumnRenamed("Rainfall_mm", "Lluvia") \
                   .withColumnRenamed("Yield_ton_per_ha", "Rendimiento")

df_renamed.printSchema()
df_renamed.show(5)

root
 |-- Crop: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Lluvia: double (nullable = true)
 |-- Temperatura: double (nullable = true)
 |-- Irrigation: string (nullable = true)
 |-- Rendimiento: double (nullable = true)

+------+--------+------+-----------+----------+-----------+
|  Crop|  Region|Lluvia|Temperatura|Irrigation|Rendimiento|
+------+--------+------+-----------+----------+-----------+
| Maize|Region_C|1485.4|       19.7|      Drip|     101.48|
|Barley|Region_D| 399.4|       29.1| Sprinkler|     127.39|
|  Rice|Region_C| 980.9|       30.5| Sprinkler|      68.99|
| Maize|Region_D|1054.3|       26.4|      Drip|     169.06|
| Maize|Region_D| 744.6|       20.4|      Drip|     118.71|
+------+--------+------+-----------+----------+-----------+
only showing top 5 rows



### 3.- Filtrado de datos(filter)

In [11]:
df_renamed.filter( (col("Crop") == "Maize") & (col("Temperatura") > 25)).show(10)

+-----+--------+------+-----------+----------+-----------+
| Crop|  Region|Lluvia|Temperatura|Irrigation|Rendimiento|
+-----+--------+------+-----------+----------+-----------+
|Maize|Region_D|1054.3|       26.4|      Drip|     169.06|
|Maize|Region_C| 846.1|       32.4|      None|      162.2|
|Maize|Region_A| 362.5|       26.6| Sprinkler|      95.23|
|Maize|Region_C|1193.3|       33.7|      None|     110.57|
|Maize|Region_C| 695.2|       27.8|     Flood|     143.84|
|Maize|Region_D|1001.4|       30.2|     Flood|     138.61|
|Maize|Region_A| 747.7|       27.7| Sprinkler|     114.58|
|Maize|Region_B|1392.9|       28.9|      Drip|     169.23|
|Maize|Region_B| 694.4|       34.7|      Drip|      96.08|
|Maize|Region_D| 848.8|       29.5|     Flood|      93.45|
+-----+--------+------+-----------+----------+-----------+
only showing top 10 rows



### 4.- Encadenamiento

In [13]:
df_renamed = (df_sel.withColumnRenamed("Temperature_C", "Temperatura")
                    .withColumnRenamed("Rainfall_mm", "Lluvia")
                    .withColumnRenamed("Yield_ton_per_ha", "Rendimiento")
                    .filter((col("Crop") == "Maize") & (col("Temperatura") > 25)).show(10)
             )

+-----+--------+------+-----------+----------+-----------+
| Crop|  Region|Lluvia|Temperatura|Irrigation|Rendimiento|
+-----+--------+------+-----------+----------+-----------+
|Maize|Region_D|1054.3|       26.4|      Drip|     169.06|
|Maize|Region_C| 846.1|       32.4|      None|      162.2|
|Maize|Region_A| 362.5|       26.6| Sprinkler|      95.23|
|Maize|Region_C|1193.3|       33.7|      None|     110.57|
|Maize|Region_C| 695.2|       27.8|     Flood|     143.84|
|Maize|Region_D|1001.4|       30.2|     Flood|     138.61|
|Maize|Region_A| 747.7|       27.7| Sprinkler|     114.58|
|Maize|Region_B|1392.9|       28.9|      Drip|     169.23|
|Maize|Region_B| 694.4|       34.7|      Drip|      96.08|
|Maize|Region_D| 848.8|       29.5|     Flood|      93.45|
+-----+--------+------+-----------+----------+-----------+
only showing top 10 rows



## Dataset 2: Lugares famosos del mundo

### 1.- Selección de datos críticos

In [15]:
places_schema = StructType([
    StructField("Place_Name", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Annual_Visitors_Millions", DoubleType(), True),
    StructField("Type", StringType(), True),
    StructField("UNESCO_World_Heritage", StringType(), True),
    StructField("Year_Built", StringType(), True),
    StructField("Entry_Fee_USD", IntegerType(), True),
    StructField("Best_Visit_Month", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Tourism_Revenue_Million_USD", LongType(), True),
    StructField("Average_Visit_Duration_Hours", DoubleType(), True),
    StructField("Famous_For", StringType(), True)
])

df_base = (spark.read
                  .format("csv")
                  .schema(places_schema)
                  .option("header", "true")
                  .load("world_famous_places_2024.csv")
                  .select("Place_Name", "Country", "UNESCO_World_Heritage", "Entry_Fee_USD", "Annual_Visitors_Millions")
            )

df_base.printSchema()

df_base.show(5, truncate=False)

root
 |-- Place_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- UNESCO_World_Heritage: string (nullable = true)
 |-- Entry_Fee_USD: integer (nullable = true)
 |-- Annual_Visitors_Millions: double (nullable = true)

+-------------------+-------------+---------------------+-------------+------------------------+
|Place_Name         |Country      |UNESCO_World_Heritage|Entry_Fee_USD|Annual_Visitors_Millions|
+-------------------+-------------+---------------------+-------------+------------------------+
|Eiffel Tower       |France       |No                   |35           |7.0                     |
|Times Square       |United States|No                   |0            |50.0                    |
|Louvre Museum      |France       |Yes                  |22           |8.7                     |
|Great Wall of China|China        |Yes                  |10           |10.0                    |
|Taj Mahal          |India        |Yes                  |15           |7.5     

### 2.- Traducción y simplificación

In [16]:
df_es = df_base.withColumnRenamed("Place_name", "Lugar") \
                .withColumnRenamed("UNESCO_World_Heritage", "Es_UNESCO") \
                .withColumnRenamed("Entry_Fee_USD", "Precio_Entrada") \
                .withColumnRenamed("Annual_Visitors_Millions", "Visitantes_Millones")

df_es.printSchema()
df_es.show(5)

root
 |-- Lugar: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Es_UNESCO: string (nullable = true)
 |-- Precio_Entrada: integer (nullable = true)
 |-- Visitantes_Millones: double (nullable = true)

+-------------------+-------------+---------+--------------+-------------------+
|              Lugar|      Country|Es_UNESCO|Precio_Entrada|Visitantes_Millones|
+-------------------+-------------+---------+--------------+-------------------+
|       Eiffel Tower|       France|       No|            35|                7.0|
|       Times Square|United States|       No|             0|               50.0|
|      Louvre Museum|       France|      Yes|            22|                8.7|
|Great Wall of China|        China|      Yes|            10|               10.0|
|          Taj Mahal|        India|      Yes|            15|                7.5|
+-------------------+-------------+---------+--------------+-------------------+
only showing top 5 rows



### 3.- Filtrado

In [22]:
df_es.filter((col("Es_UNESCO") == "Yes") & (col ("Precio_Entrada") <= 20)).show(5)

+--------------------+-------+---------+--------------+-------------------+
|               Lugar|Country|Es_UNESCO|Precio_Entrada|Visitantes_Millones|
+--------------------+-------+---------+--------------+-------------------+
| Great Wall of China|  China|      Yes|            10|               10.0|
|           Taj Mahal|  India|      Yes|            15|                7.5|
|           Colosseum|  Italy|      Yes|            18|               7.65|
|      Forbidden City|  China|      Yes|             8|                9.0|
|Notre-Dame Cathedral| France|      Yes|             0|               13.0|
+--------------------+-------+---------+--------------+-------------------+
only showing top 5 rows



## Dataset 3: Registro turístico de Castilla y León

### 1.- Selección y saneamiento

In [34]:
turismo_schema = StructType([
    StructField("establecimiento", StringType(), True),
    StructField("n_registro", StringType(), True),
    StructField("codigo", StringType(), True),
    StructField("tipo", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("especialidades", StringType(), True),
    StructField("clase", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("direccion", StringType(), True),
    StructField("c_postal", StringType(), True), 
    StructField("provincia", StringType(), True),
    StructField("municipio", StringType(), True),
    StructField("localidad", StringType(), True),
    StructField("nucleo", StringType(), True),
    StructField("telefono_1", StringType(), True),
    StructField("telefono_2", StringType(), True),
    StructField("telefono_3", StringType(), True),
    StructField("email", StringType(), True),
    StructField("web", StringType(), True),
    StructField("q_calidad", StringType(), True),
    StructField("posada_real", StringType(), True),
    StructField("plazas", DoubleType(), True),
    StructField("gps_longitud", DoubleType(), True),
    StructField("gps_latitud", DoubleType(), True),
    StructField("accesible_a_personas_con_discapacidad", StringType(), True),
    StructField("column_27", StringType(), True),
    StructField("posicion", StringType(), True)
])

df_contactos = (spark.read
                .format("csv")
                .schema(turismo_schema)
                .option("header", "true")
                .option("sep", ";")
                .load("registro-de-turismo-de-castilla-y-leon.csv")
                .select("nombre", "tipo", "provincia", "web", "email")
             )

df_contactos.printSchema()

df_contactos.show(5)

root
 |-- nombre: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- web: string (nullable = true)
 |-- email: string (nullable = true)

+--------------------+--------------------+---------+--------------------+--------------------+
|              nombre|                tipo|provincia|                 web|               email|
+--------------------+--------------------+---------+--------------------+--------------------+
|BERNARDO MORO MEN...|Profesional de Tu...| Asturias|                NULL|bernardomoro@hotm...|
|        LA SASTRERÍA|Casa Rural de Alq...|    Ávila|www.lasastreriade...|                NULL|
|         LAS HAZANAS|Casa Rural de Alq...|    Ávila|                NULL|lashazanas@hotmai...|
| LA CASITA DEL PAJAR|Casa Rural de Alq...|    Ávila|                NULL|lashazanas@hotmai...|
|            MARACANA|                 Bar|    Ávila|                NULL|emo123anatoliev@g...|
+--------------------+-----------------

### 2.- Renombrado estándar

In [36]:
df_limpio = df_contactos.withColumnRenamed("nombre", "nombre_establecimiento") \
                .withColumnRenamed("tipo", "categoria_actividad") \
                .withColumnRenamed("web", "sitio_web") \
                .withColumnRenamed("email", "correo_electronico")

df_limpio.printSchema()
df_limpio.show(5)

root
 |-- nombre_establecimiento: string (nullable = true)
 |-- categoria_actividad: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- sitio_web: string (nullable = true)
 |-- correo_electronico: string (nullable = true)

+----------------------+--------------------+---------+--------------------+--------------------+
|nombre_establecimiento| categoria_actividad|provincia|           sitio_web|  correo_electronico|
+----------------------+--------------------+---------+--------------------+--------------------+
|  BERNARDO MORO MEN...|Profesional de Tu...| Asturias|                NULL|bernardomoro@hotm...|
|          LA SASTRERÍA|Casa Rural de Alq...|    Ávila|www.lasastreriade...|                NULL|
|           LAS HAZANAS|Casa Rural de Alq...|    Ávila|                NULL|lashazanas@hotmai...|
|   LA CASITA DEL PAJAR|Casa Rural de Alq...|    Ávila|                NULL|lashazanas@hotmai...|
|              MARACANA|                 Bar|    Ávila|                

### 3.- Filtrado de texto

In [39]:
df_limpio.filter(
                (col("provincia") == "Burgos") &
                (col("categoria_actividad").like("%Bodegas%")) & 
                (col("sitio_web").isNotNull())
).show(5)

+----------------------+--------------------+---------+--------------------+--------------------+
|nombre_establecimiento| categoria_actividad|provincia|           sitio_web|  correo_electronico|
+----------------------+--------------------+---------+--------------------+--------------------+
|        BODEGAS TARSUS|g - Bodegas y los...|   Burgos|  www.tarsusvino.com|                NULL|
|  BODEGAS DOMINIO D...|g - Bodegas y los...|   Burgos|www.dominiodecair...|bodegas@dominiode...|
|    TERRITORIO LUTHIER|g - Bodegas y los...|   Burgos|territorioluthier...|luthier@territori...|
|    BODEGA COVARRUBIAS|g - Bodegas y los...|   Burgos| http://valdable.com|   info@valdable.com|
|  BODEGAS PASCUAL, ...|g - Bodegas y los...|   Burgos|222.bodegaspascua...|export@bodegaspas...|
+----------------------+--------------------+---------+--------------------+--------------------+
only showing top 5 rows



In [41]:
sc.stop()